# Introvert vs Extrovert Classification

This notebook trains an explainable, high-accuracy model to predict whether a person is an introvert or an extrovert using the provided dataset.

## Imports

Load libraries used for data prep, modeling, and evaluation.

In [7]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [8]:
data_path = Path('personality_dataset.csv')
df = pd.read_csv(data_path)
df.head()

,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,4.0,No,4.0,6.0,No,13.0,5.0,Extrovert
1,9.0,Yes,0.0,0.0,Yes,0.0,3.0,Introvert
2,9.0,Yes,1.0,2.0,Yes,5.0,2.0,Introvert
3,0.0,No,6.0,7.0,No,14.0,8.0,Extrovert
4,3.0,No,9.0,4.0,No,8.0,5.0,Extrovert


In [9]:
df.isna().sum()

Time_spent_Alone             63
Stage_fear                   73
Social_event_attendance      62
Going_outside                66
Drained_after_socializing    52
Friends_circle_size          77
Post_frequency               65
Personality                   0
dtype: int64

In [10]:
target_col = 'Personality'
X = df.drop(columns=[target_col])
y = df[target_col]

categorical_features = ['Stage_fear', 'Drained_after_socializing']
numeric_features = [col for col in X.columns if col not in categorical_features]

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    clf = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    f1 = f1_score(y_test, preds, pos_label='Introvert')
    acc = accuracy_score(y_test, preds)
    print(f'[{name}] Accuracy: {acc:.3f} | F1 (Introvert): {f1:.3f}')
    return clf, acc, f1

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
rf_model = RandomForestClassifier(
    n_estimators=300, random_state=42, class_weight='balanced'
 )

lr_clf, lr_acc, lr_f1 = evaluate_model('Logistic Regression', lr_model, X_train, X_test, y_train, y_test)
rf_clf, rf_acc, rf_f1 = evaluate_model('Random Forest', rf_model, X_train, X_test, y_train, y_test)

[Logistic Regression] Accuracy: 0.917 | F1 (Introvert): 0.917
[Random Forest] Accuracy: 0.902 | F1 (Introvert): 0.900


In [12]:
best_clf = rf_clf if rf_f1 >= lr_f1 else lr_clf
best_name = 'Random Forest' if rf_f1 >= lr_f1 else 'Logistic Regression'

print(f'Using best model: {best_name}')
preds = best_clf.predict(X_test)
print('Confusion matrix:')
print(confusion_matrix(y_test, preds))
print('Classification report:')
print(classification_report(y_test, preds))

Using best model: Logistic Regression
Confusion matrix:
[[267  31]
 [ 17 265]]
Classification report:
              precision    recall  f1-score   support

   Extrovert       0.94      0.90      0.92       298
   Introvert       0.90      0.94      0.92       282

    accuracy                           0.92       580
   macro avg       0.92      0.92      0.92       580
weighted avg       0.92      0.92      0.92       580



In [13]:
feature_names = best_clf.named_steps['preprocess'].get_feature_names_out()
model = best_clf.named_steps['model']

if hasattr(model, 'coef_'):
    coef = model.coef_[0]
    top_idx = np.argsort(np.abs(coef))[::-1][:10]
    top_features = pd.DataFrame({
        'feature': feature_names[top_idx],
        'coefficient': coef[top_idx]
    })
    top_features
elif hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:10]
    top_features = pd.DataFrame({
        'feature': feature_names[top_idx],
        'importance': importances[top_idx]
    })
    top_features

In [14]:
models_dir = Path('models')
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / 'personality_model.joblib'
metadata_path = models_dir / 'metadata.json'

joblib.dump(best_clf, model_path)

metadata = {
    'model_name': best_name,
    'features': X.columns.tolist(),
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'classes': sorted(y.unique().tolist())
}

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

model_path, metadata_path

(PosixPath('models/personality_model.joblib'),
 PosixPath('models/metadata.json'))

In [18]:
# Prediction demo using saved artifacts
import json
from pathlib import Path

import joblib
import pandas as pd

models_dir = Path("models")
model_path = models_dir / "personality_model.joblib"
metadata_path = models_dir / "metadata.json"

model = joblib.load(model_path)
with metadata_path.open("r", encoding="utf-8") as f:
    meta = json.load(f)

sample = {
    "Time_spent_Alone": 0,
    "Stage_fear": "No",
    "Social_event_attendance": 7.0,
    "Going_outside": 6.0,
    "Drained_after_socializing": "No",
    "Friends_circle_size": 10.0,
    "Post_frequency": 4.0,
}

df = pd.DataFrame([{k: sample[k] for k in meta["features"]}])
proba = model.predict_proba(df)
pred = model.classes_[proba.argmax(axis=1)][0]

print("Prediction:", pred)
print("Confidence:", float(proba.max()))

Prediction: Extrovert
Confidence: 0.9249456720732174


## Deployment Notes

The saved model and metadata files are used by the AWS Lambda inference handler. The API will accept JSON with the same feature names as the dataset (excluding the target Personality).